# Controlled Replication: YOLOv8x on the External fy4c2 Dataset

**Purpose:** Train the merged-ball detector's exact YOLOv8x configuration on an
external, unmodified, 7-class public dataset, to test whether ball-detection difficulty
on the evaluation clips is data-driven (specific to the merged training set) or
architecture-driven (a property of YOLOv8x on this class of footage).  
**Inputs:** Roboflow dataset `basketball-players-fy4c2-vfsuv` v17 (classes Ball, Clock,
Hoop, Overlay, Player, Ref, Scoreboard); `yolov8x.pt` base weights; hyperparameters
copied unchanged from `train_ball_detector.ipynb`.  
**Outputs:** the `runs/detect/ball_fy4c2_replication/` training run; its best checkpoint
saved as `models/ball_fy4c2_replication.pt`.  
**Backs:** `results/training/ball_fy4c2_replication/` and `models/ball.pt`: the
checkpoint this notebook produced was adopted as the pipeline's production detector
under that name.

This is the production detector's training record. The dataset is used exactly as
downloaded: no class filtering, no merging, no targeted ball-image augmentation, so the
only variable that differs from the merged-ball run is the training data, which is what
makes the comparison controlled.

At run time this notebook writes only `models/ball_fy4c2_replication.pt`, with an
explicit guard (Section 7) against writing to `models/ball.pt` or
`models/ball_baseline.pt`; the adoption under the production name happened outside this
notebook. It does not modify `training/train_ball_detector.ipynb`,
`basketball/detection/ball_detector.py` or `basketball/detection/ball_interpolation.py`.

Run from the repo root on JupyterHub, kernel "Python (Basketball Analytics)". The first
cell pins the working directory to the repo root.

In [1]:
import os
from pathlib import Path

# Same working-directory pin used by train_ball_detector.ipynb: this notebook lives in
# training/, so a kernel launched there resolves every relative path one level too deep.
if Path.cwd().name == 'training':
    os.chdir('..')

repo_root = Path.cwd()
assert (repo_root / 'training').is_dir() and (repo_root / 'basketball').is_dir(), (
    f'Unexpected working directory: {repo_root}. '
    f'Launch this notebook from the repo root or from training/.'
)
print(f'Working directory: {repo_root}')

Working directory: /home/jovyan/nba-video-analytics


## 1. Environment check

Confirms the GPU and the Ultralytics install. The settings-reset warning in the output
is an Ultralytics first-run notice and harmless.

In [2]:
import torch
import ultralytics

print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'Ultralytics: {ultralytics.__version__}')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available — verify kernel is set to Python (Basketball Analytics) before proceeding.')

WARNING ⚠️ Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at '/home/jovyan/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.4.0+cu124 | CUDA: True | GPU: NVIDIA A40
Ultralytics: 8.4.62


## 2. Download dataset

Downloads the external **basketball-players-fy4c2-vfsuv** dataset (v17, 7 classes) from
Roboflow in YOLOv8 format. This is a public dataset unrelated to the project's merged
training set, used here as the controlled comparison point.

Export your personal Roboflow API key as `ROBOFLOW_API_KEY` before running the download
cell below. **Never hardcode or commit a real key**: the cell below reads it from the
environment.

In [4]:
from pathlib import Path

dataset_dir = Path('training/fy4c2-replication')
dataset_dir.mkdir(parents=True, exist_ok=True)
print(f'Dataset directory: {dataset_dir.resolve()}')

Dataset directory: /home/jovyan/nba-video-analytics/training/fy4c2-replication


In [5]:
# Reads the Roboflow API key from the ROBOFLOW_API_KEY environment variable:
# export it in the kernel environment before running; never hardcode a key here.
# Workspace/project/version identify the external fy4c2 dataset exactly as specified
# for this replication; location/overwrite kwargs follow the same download-target
# convention as train_ball_detector.ipynb (download into a project-local folder under
# training/, not the repo root).
import os

from roboflow import Roboflow

rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
project = rf.workspace('workspace-5ujvu').project('basketball-players-fy4c2-vfsuv')
version = project.version(17)
dataset = version.download('yolov8', location=str(dataset_dir), overwrite=True)

print(f'Download complete: {dataset.location}')

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to training/fy4c2-replication in yolov8:: 100%|██████████| 652/652 [00:00<00:00, 6563.79it/s]

Download complete: /home/jovyan/nba-video-analytics/training/fy4c2-replication


## 3. Dataset verification

Reads the downloaded `data.yaml` directly to get the **actual** class-to-index mapping
for this dataset, not an assumption from the dataset's documentation. Confirms all 7
classes are present and unfiltered, and reports per-split image and per-class
annotation counts.

In [6]:
import yaml
from collections import Counter
from pathlib import Path

data_yaml_path = dataset_dir / 'data.yaml'
with open(data_yaml_path) as f:
    data_config = yaml.safe_load(f)

raw_names = data_config['names']
class_names = [raw_names[i] for i in sorted(raw_names)] if isinstance(raw_names, dict) else list(raw_names)

print('Dataset configuration (read directly from data.yaml):')
print(f'  Classes (nc): {data_config["nc"]}')
print(f'  Class names:  {class_names}')

assert data_config['nc'] == len(class_names) == 7, (
    f'Expected exactly 7 classes (unmodified fy4c2 dataset), got nc={data_config["nc"]}, '
    f'names={class_names}. This experiment requires the dataset unmodified — investigate '
    f'before proceeding.'
)

ball_class_index = next(i for i, name in enumerate(class_names) if name.lower() == 'ball')
print(f'\n  -> "ball" class confirmed at index {ball_class_index} (read from data.yaml, not assumed).')

splits = ['train', 'valid', 'test']
total_images = 0
total_annotations_by_class = Counter()

print('\nPer-split counts (read from label files; all 7 classes retained, unmodified):')
for split in splits:
    images_dir = dataset_dir / split / 'images'
    labels_dir = dataset_dir / split / 'labels'

    image_count = sum(
        1 for image_path in images_dir.glob('*')
        if image_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    ) if images_dir.is_dir() else 0

    split_counts = Counter()
    if labels_dir.is_dir():
        for label_path in labels_dir.glob('*.txt'):
            for line in label_path.read_text().splitlines():
                if not line.strip():
                    continue
                class_id = int(line.split()[0])
                split_counts[class_id] += 1

    total_images += image_count
    total_annotations_by_class.update(split_counts)

    per_class_str = ', '.join(
        f'{class_names[cid]}={count}' for cid, count in sorted(split_counts.items())
    )
    print(f'  {split:5s}: {image_count:5d} images | {per_class_str}')

print(f'\nTotals: {total_images} images across all splits.')
print('Total annotations per class:')
for cid in sorted(total_annotations_by_class):
    print(f'  {class_names[cid]:12s} (index {cid}): {total_annotations_by_class[cid]}')

Dataset configuration (read directly from data.yaml):
  Classes (nc): 7
  Class names:  ['Ball', 'Clock', 'Hoop', 'Overlay', 'Player', 'Ref', 'Scoreboard']

  -> "ball" class confirmed at index 0 (read from data.yaml, not assumed).

Per-split counts (read from label files; all 7 classes retained, unmodified):
  train:   256 images | Ball=213, Clock=249, Hoop=203, Overlay=294, Player=2021, Ref=495, Scoreboard=211
  valid:    32 images | Ball=28, Clock=35, Hoop=30, Overlay=40, Player=259, Ref=64, Scoreboard=27
  test :    32 images | Ball=24, Clock=37, Hoop=25, Overlay=36, Player=260, Ref=61, Scoreboard=28

Totals: 320 images across all splits.
Total annotations per class:
  Ball         (index 0): 265
  Clock        (index 1): 321
  Hoop         (index 2): 258
  Overlay      (index 3): 370
  Player       (index 4): 2540
  Ref          (index 5): 620
  Scoreboard   (index 6): 266


## 4. Train YOLOv8x

Fine-tunes YOLOv8x on the fy4c2 dataset exactly as downloaded: all 7 classes, no
class deletion, no targeted ball-image augmentation. This is deliberate. The
hyperparameters below are copied unchanged from `train_ball_detector.ipynb`, so the
only variable that differs between that run and this one is the training data;
adding augmentation here would confound the comparison.

Configuration: 100-epoch ceiling with early stopping (patience=30), batch size 32,
imgsz=640, Ultralytics' default seed=0. The run name `ball_fy4c2_replication` keeps
this run's output separate from `ball_train_merged`. The IProgress message in the
output is a Jupyter display notice and harmless.

In [7]:
from ultralytics import YOLO
from pathlib import Path

data_yaml = str((dataset_dir / 'data.yaml').resolve())

model = YOLO('yolov8x.pt')

# No explicit project= argument: matches train_ball_detector.ipynb's avoidance of the
# path-doubling bug. Ultralytics defaults the output to runs/detect/ball_fy4c2_replication/.
results = model.train(
    data=data_yaml,
    epochs=100,
    patience=30,
    batch=32,
    imgsz=640,
    name='ball_fy4c2_replication',
)

print(f'\nTraining complete. Best weights: {results.save_dir}/weights/best.pt')

New https://pypi.org/project/ultralytics/8.4.95 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.4.0+cu124 CUDA:0 (NVIDIA A40, 45619MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/nba-video-analytics/training/fy4c2-replication/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 634.6±247.7 MB/s, size: 60.4 KB)
val: Scanning /home/jovyan/nba-video-analytics/training/fy4c2-replication/valid/labels... 32 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 32/32 511.8it/s 0.1s
val: New cache created: /home/jovyan/nba-video-analytics/training/fy4c2-replication/valid/labels.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000909, momentum=0.9) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0005), 103 bias(decay=0.0)
Plotting labels to /home/jovyan/nba-video-analytics/runs/detect/ball_fy4c2_replication/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader

## 5. Aggregate metrics (dataset level)

Reports final validation metrics read directly from the training run's `results.csv`
on disk (located via glob), not from the in-memory `results` variable, so they cannot
be stale from a previous kernel session, the same practice as
`train_ball_detector.ipynb`.

These metrics are aggregated across all 7 classes, since that is what Ultralytics
writes to `results.csv`. Section 6 below isolates the ball class specifically, which
is what this experiment actually needs to answer the research question.

In [8]:
import glob
import pandas as pd
from pathlib import Path

run_dirs = glob.glob('runs/**/ball_fy4c2_replication*/', recursive=True)
if not run_dirs:
    raise FileNotFoundError('No ball_fy4c2_replication run directory found under runs/.')

latest_run = Path(sorted(run_dirs)[-1])
results_csv = latest_run / 'results.csv'

history = pd.read_csv(results_csv)
history.columns = history.columns.str.strip()

# best.pt is chosen by Ultralytics fitness (weighted mAP); approximate it here
# by the epoch with the highest mAP50-95 and report that epoch's metrics.
best_row = history.loc[history['metrics/mAP50-95(B)'].idxmax()]

print(f'Metrics read from: {results_csv}')
print(f'Epochs logged:     {len(history)} (best epoch: {int(best_row["epoch"])})')
print('\nFinal validation metrics (best epoch, matches best.pt, aggregated across all 7 classes):')
print(f'  Precision:  {best_row["metrics/precision(B)"]:.4f}')
print(f'  Recall:     {best_row["metrics/recall(B)"]:.4f}')
print(f'  mAP50:      {best_row["metrics/mAP50(B)"]:.4f}')
print(f'  mAP50-95:   {best_row["metrics/mAP50-95(B)"]:.4f}')

Metrics read from: runs/detect/ball_fy4c2_replication/results.csv
Epochs logged:     100 (best epoch: 99)

Final validation metrics (best epoch, matches best.pt, aggregated across all 7 classes):
  Precision:  0.9669
  Recall:     0.8949
  mAP50:      0.9434
  mAP50-95:   0.7744


## 6. Per-class metrics: isolating the ball class

`results.csv` only reports dataset-level aggregate metrics, blended across all 7
classes. Since the research question is specifically about ball-detection
performance, this cell runs a fresh validation pass (`model.val()`, not a reused
in-memory object from Section 4's training call) against the best checkpoint and
reads Ultralytics' per-class arrays (`metrics.box.ap_class_index`, `.p`, `.r`,
`.ap50`, `.maps`) to isolate the ball class, identified again from `model.names`,
not assumed to match Section 3's index (it should match; this is a second
independent confirmation from the checkpoint itself rather than the dataset file).

In [9]:
import glob
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

checkpoints = glob.glob('runs/**/ball_fy4c2_replication*/weights/best.pt', recursive=True)
if not checkpoints:
    raise FileNotFoundError('No ball_fy4c2_replication checkpoint found under runs/.')
best_pt = sorted(checkpoints)[-1]

val_model = YOLO(best_pt)
print(f'Loaded checkpoint: {best_pt}')
print(f'Class mapping from checkpoint: {val_model.names}')

val_metrics = val_model.val(data=data_yaml, split='test')

per_class_rows = []
for idx, class_id in enumerate(val_metrics.box.ap_class_index):
    class_id = int(class_id)
    per_class_rows.append({
        'class_id': class_id,
        'class_name': val_model.names[class_id],
        'precision': float(val_metrics.box.p[idx]),
        'recall': float(val_metrics.box.r[idx]),
        'mAP50': float(val_metrics.box.ap50[idx]),
        'mAP50-95': float(val_metrics.box.maps[class_id]),
    })

per_class_df = pd.DataFrame(per_class_rows).set_index('class_id').sort_index()
print('\nPer-class validation metrics (test split):')
print(per_class_df)

ball_row = per_class_df[per_class_df['class_name'].str.lower() == 'ball']
print('\nBall-class metrics — the class of interest for this experiment:')
print(ball_row)

Loaded checkpoint: runs/detect/ball_fy4c2_replication/weights/best.pt
Class mapping from checkpoint: {0: 'Ball', 1: 'Clock', 2: 'Hoop', 3: 'Overlay', 4: 'Player', 5: 'Ref', 6: 'Scoreboard'}
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.4.0+cu124 CUDA:0 (NVIDIA A40, 45619MiB)
Model summary (fused): 113 layers, 68,130,309 parameters, 0 gradients, 257.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1502.6±759.0 MB/s, size: 73.3 KB)
val: Scanning /home/jovyan/nba-video-analytics/training/fy4c2-replication/test/labels... 32 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 32/32 851.4it/s 0.0s
val: New cache created: /home/jovyan/nba-video-analytics/training/fy4c2-replication/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.2s/it 2.4s4.3s
                   all         32        471      0.973      0.954      0.962      0.789
                  Ball         24         24      0.933      0.792      0

## 7. Save checkpoint

Saves the best checkpoint as `models/ball_fy4c2_replication.pt`. An explicit guard
below refuses to write to `models/ball.pt` or `models/ball_baseline.pt` under any
circumstance; the run must never overwrite either existing checkpoint.

In [10]:
import shutil
import glob
from pathlib import Path

checkpoints = glob.glob('runs/**/ball_fy4c2_replication*/weights/best.pt', recursive=True)
if not checkpoints:
    raise FileNotFoundError('No checkpoint found — check runs/ for the training output directory.')

best_pt = sorted(checkpoints)[-1]
size_mb = Path(best_pt).stat().st_size / (1024 * 1024)
print(f'Best checkpoint: {best_pt} ({size_mb:.1f} MB)')

dest_path = Path('models/ball_fy4c2_replication.pt')
protected_paths = {Path('models/ball.pt'), Path('models/ball_baseline.pt')}
assert dest_path not in protected_paths, (
    f'Refusing to write to {dest_path} — this experiment must never overwrite a production checkpoint.'
)

dest_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt, dest_path)
print(f'Saved replication checkpoint to {dest_path} ({dest_path.stat().st_size / (1024 * 1024):.1f} MB)')
print('models/ball.pt and models/ball_baseline.pt were not written to by this notebook.')

Best checkpoint: runs/detect/ball_fy4c2_replication/weights/best.pt (130.4 MB)
Saved replication checkpoint to models/ball_fy4c2_replication.pt (130.4 MB)
models/ball.pt and models/ball_baseline.pt were not written to by this notebook.


## 8. Outcome

The run produced `runs/detect/ball_fy4c2_replication/weights/best.pt`, saved as
`models/ball_fy4c2_replication.pt` with the Section 7 guard keeping the run away from
the existing checkpoints. That checkpoint was adopted as the pipeline's production
detector under the `models/ball.pt` name, serving both player and ball detection.

The ball class index is confirmed programmatically in Section 3 (from `data.yaml`)
and re-confirmed in Section 6 (from the trained checkpoint's own `model.names`); it
is never assumed to be 0. Training curves and validation artefacts ship in
`results/training/ball_fy4c2_replication/`.